In [ ]:
import os
import sys
import gc
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
from config import DATA_PATH


In [ ]:
fnames_probs = {
    '2024b': '2024BehaviorAgentProbabilities.csv',
    # '2025a': '2025AllAgentProbabilities.csv',
    # '2025b': '2025BehaviorAgentProbabilities.csv'
}
fnames_dfs = {
    '2024': 'dfHourly2024.csv',
    # '2025': 'dfHourly2025.csv'
}
probs = {k: pd.read_csv(os.path.join(DATA_PATH, v)) for k, v in fnames_probs.items()}
dfs = {k: pd.read_csv(os.path.join(DATA_PATH, v)) for k, v in fnames_dfs.items()}
# msgs = pd.read_csv(os.path.join(DATA_PATH, 'IHSMessagePool.csv'))

del fnames_probs, fnames_dfs
gc.collect()

# 2024

## Variables Needed
* Contextual variables 
* Intervention & Probability
    * Send vs no send: `action_id` = {1: 'Send', 0: 'No Send'}
    * Domain: `message_metric`
    * Chosen message: `core_message_id`
    * Message strategy: `strategy`
    * uniform for send vs no send: `p_send`
    * uniform for strategy & domain of chosen message (without replacement!): `p_mood`, `p_steps`, `p_sleep`
        * these are conditional probabilities P(Domain|Action=1)
* BL Characteristics

### Intervention & Probability

In [ ]:
interventions = probs['2024b'].copy()
p_cols = interventions.columns[interventions.columns.str.startswith('p_')]

# convert to datetime
interventions['recorded_time_utc'] = pd.to_datetime(interventions['recorded_time_utc'], utc=True)
interventions['hour_utc'] = pd.to_datetime(interventions['hour_utc'], utc=True)
interventions['decision_date'] = pd.to_datetime(interventions['decision_date'])

# verify decision_date reflects time in UTC
assert (interventions['decision_date'] == interventions['hour_utc'].dt.date).all()

# verify Preinternship notification => before July 1
p = interventions['notification_id'].str.extract(r"(\D+)\d*(?:-\d+-\d)?$")[0] == 'Preinternship'
q = interventions['recorded_time_utc'] < pd.Timestamp("2024-07-01", tz="UTC")
assert (~p | q).all()
# verify after July 1 => not Preinternship notification
assert (q | ~p).all()

# verify NoMessage notification <=> action_id == 0
p = interventions['notification_id'].str.extract(r"(\D+)\d*(?:-\d+-\d)?$")[0] == 'NoMessage'
q = interventions['action_id'] == 0
assert (~p | q).all()
assert (~q | p).all()

# filter out preinternship
interventions = interventions.loc[interventions['recorded_time_utc'] >= pd.Timestamp("2024-07-01", tz="UTC")]

# verify action_id == 1 <=> core_message_id
assert (~(interventions['action_id'] == 1) | (~interventions['core_message_id'].isna())).all()
assert (interventions['core_message_id'].isna() | (interventions['action_id'] == 1)).all()

# verify probability is not missing
assert interventions[p_cols].notnull().all().all()

# verify there is only 1 notification per day per participant
s = interventions\
    .groupby('participant_id')['decision_date']\
    .apply(lambda s: s.duplicated().sum())
assert (s == 0).all()

# verify no send <=> missing metric & missing strategy
p = interventions['action_id'] == 0
q = interventions['message_metric'].isnull() & interventions['strategy'].isnull()
assert (~p | q).all()
assert (~q | p).all()

# verify chosen message has consistent domain and strategy
assert interventions\
    .groupby('core_message_id')[['message_metric', 'strategy']]\
    .nunique().eq(1).all(axis=1).all()

# verify probabilities add up to one
assert np.isclose(interventions[['p_send', 'p_no_send']].sum(axis=1), 1).all()
assert np.isclose(interventions[['p_mood', 'p_steps', 'p_sleep']].sum(axis=1), 1).all()
assert np.isclose(interventions[
    p_cols.drop(['p_send', 'p_no_send', 'p_mood', 'p_steps', 'p_sleep'])
].sum(axis=1), 1).all()

# verify p_send = p_no_send = 0.5
assert np.isclose(interventions[['p_send', 'p_no_send']], 0.5).all().all()

# subset to relevant columns
interventions = interventions[[
    'decision_id',
    'participant_id',
    'decision_date',
    'hour_utc',
    'action_id',
    'message_metric',
    'strategy',
    'p_send',
    'p_mood',
    'p_steps',
    'p_sleep'
]]\
.set_index('decision_id')\
.rename(columns={
    'hour_utc': 'decision_time_utc',
    'decision_date': 'decision_date_utc'
})

# export
interventions.to_csv(os.path.join(DATA_PATH, "processed/interventions_2024.csv"))

del interventions, p, q, s
gc.collect()

### Contextual Variables & Outcomes

In [ ]:
def wide2long(df, regex=r'^(Fitbit|Garmin|Healthkit)', extra=('Mood',), pid='participant_id'):
    """Device columns -> long: one row per index entry and device, one column per stream.

    Keeps all of `df`'s index levels and adds `device_type`. Columns in
    `extra` (e.g. Mood) are not device-specific: they are copied onto every
    device the participant (`pid` index level) has data from, including
    rows where that device recorded nothing.
    """
    keys = list(df.index.names)

    dev = df.filter(regex=regex)
    dev.columns = pd.MultiIndex.from_tuples(
        [c.split('_', 1) for c in dev.columns], names=['device_type', 'outcome']
    )
    long = dev.stack('device_type').dropna(how='all').reset_index()

    extra = [c for c in extra if c in df.columns]
    if extra:
        devices = long[[pid, 'device_type']].drop_duplicates()   # each participant's devices
        ext = df[extra].dropna(how='all').reset_index().merge(devices, on=pid)
        long = long.merge(ext, on=[*keys, 'device_type'], how='outer')

    return long.set_index([*keys, 'device_type']).sort_index()


contexts = dfs['2024'].copy()
contexts = contexts.rename(columns={
    'time_utc': 'context_time_utc',
    'time_local': 'context_time_local'
})
# get date
contexts['context_time_utc'] = pd.to_datetime(contexts['context_time_utc'])
# filter out data before 7/1
contexts = contexts.loc[contexts['context_time_utc'] >= pd.Timestamp("2024-07-01", tz="UTC")]
# replace 0 mood with NaN
contexts['Mood'] = contexts['Mood'].replace(0, np.nan)

# convert from wide to long
idx = ['participant_id', 'context_time_utc', 'context_time_local']
contexts_wide = contexts.set_index(idx)
contexts_long = wide2long(contexts_wide)
del contexts, contexts_wide
gc.collect()

# combined Healthkit: watch, NaNs filled with phone
watch = contexts_long.xs('HealthkitWatch', level='device_type')
phone = contexts_long.xs('HealthkitPhone', level='device_type')
combined = (
    pd.concat({'Healthkit': watch.combine_first(phone)}, names=['device_type'])
    .reorder_levels([*idx, 'device_type'])
)
contexts_long = pd.concat([contexts_long, combined])

# export
contexts_long.to_csv(os.path.join(DATA_PATH, "processed/contexts_2024.csv"))

del watch, phone, combined, contexts_long
gc.collect()

In [ ]:
# column suffix -> aggregation (first match wins; default is mean)
RULES = {
    "steps": "sum",           # total
    "sleep_minutes": "sum",   # total minutes asleep
    "hr_avg": "mean",
    "hr_max": "max",
    "RHR": "median",          # usually a daily value repeated across hours
    "Mood": "mean",
    "_from_phone": "mean",    # share of hours filled from phone
}
 
# columns that also get a within-window SD
SD_SUFFIXES = ("hr_avg", "Mood")
 
 
def _rule(col: str) -> str:
    return next((how for suffix, how in RULES.items() if col.endswith(suffix)), "mean")
 
 # TODO: fix bug in aggregate
def aggregate(
    df: pd.DataFrame,
    keys: list[str] = ("participant_id", "decision_date_utc"),
    min_step_hours: int | None = None,
) -> pd.DataFrame:
    """Aggregate hourly rows to one row per group, one rule per data stream.
 
    Adds `<col>_sd` = SD of hourly values for each hr_avg and Mood column
    (NaN if fewer than 2 values), and `<col>_n` = number of non-null hours
    per group for each stream. Groups with no data for a stream are NaN
    (not 0, which plain sum gives). If `min_step_hours` is set, step totals
    from groups with fewer hours of step data are set to NaN, since sums
    from partial windows are undercounts.
    """
    keys = list(keys)
    g = df.groupby(keys)
    n = g.count()
    cols = [c for c in df.columns if c not in keys]  # keys may be columns or index levels
    out = g.agg({c: _rule(c) for c in cols}).where(n > 0)
 
    if min_step_hours is not None:
        steps = [c for c in out.columns if c.endswith("steps")]
        out[steps] = out[steps].where(n[steps] >= min_step_hours)
 
    # within-window variability
    sd_cols = [c for c in cols if c.endswith(SD_SUFFIXES)]
    sd = g[sd_cols].std().add_suffix("_sd")
 
    counts = n[[c for c in n.columns if not c.endswith("_from_phone")]]
    return out.join(sd).join(counts.add_suffix("_n"))
 
 
def hour_utc(s: pd.Series) -> pd.Series:
    """Parse to UTC timestamps and floor to the start of the hour."""
    return pd.to_datetime(s, utc=True).dt.floor("h")
 
 
def contexts_between_decisions(
    contexts: pd.DataFrame,
    interventions: pd.DataFrame,
    streams: list[str],
    by: list[str] = ("participant_id", "device_type"),
    ctx_time: str = "context_time_utc",
    dec_time: str = "decision_time_utc",
) -> pd.DataFrame:
    """Aggregate hourly contexts between consecutive decision points.
 
    Times are UTC datetimes floored to the hour. Context hours c with
    t_prev <= c < t are aggregated for decision t. Each participant's first
    decision has no t_prev and gets all context hours c < t.
 
    Returns `aggregate`'s features indexed by `by` + `dec_time`, plus
    `window_h` = t - t_prev in hours (NaN for the first decision; its `_n`
    counts show how much data it covers). To join back, merge on
    [pid, dec_time] after interventions[dec_time] = hour_utc(interventions[dec_time]).
    """
    if ctx_time == dec_time:
        raise ValueError("ctx_time and dec_time must have different names.")
    by = list(by)
    pid = by[0]
 
    ctx = contexts.assign(**{ctx_time: hour_utc(contexts[ctx_time])}).sort_values(ctx_time)
 
    intv = (
        interventions.assign(**{dec_time: hour_utc(interventions[dec_time])})
        .sort_values([pid, dec_time])
    )
 
    # checks: complete, unique decision times
    assert intv[dec_time].notna().all(), "missing decision times"
    assert not intv.duplicated([pid, dec_time]).any(), "duplicate decision times per participant"
 
    intv["t_prev"] = intv.groupby(pid)[dec_time].shift()
    intv["window_h"] = (intv[dec_time] - intv["t_prev"]) / pd.Timedelta("1h")
 
    # check: missing t_prev <=> participant's first (earliest) decision
    is_first = intv.groupby(pid).cumcount() == 0
    earliest = intv.groupby(pid)[dec_time].transform("min")
    assert is_first.sum() == intv[pid].nunique()
    assert (intv["t_prev"].isna() == is_first).all()
    assert (intv.loc[is_first, dec_time] == earliest[is_first]).all()
 
    intv = intv.sort_values(dec_time)
    intv[dec_time] = intv[dec_time].astype(ctx[ctx_time].dtype)  # matching resolution
 
    # assign each context hour to the next decision: t_prev <= c < t
    ctx = pd.merge_asof(
        ctx, intv, left_on=ctx_time, right_on=dec_time, by=pid,
        direction="forward", allow_exact_matches=False,
    ).dropna(subset=[dec_time])   # drops hours after the last decision
 
    feats = aggregate(ctx[[*by, dec_time, *streams]], keys=[*by, dec_time])
    return feats.join(intv.set_index([pid, dec_time]))
 

In [ ]:
interventions = pd.read_csv(os.path.join(DATA_PATH, "processed/interventions_2024.csv"))
contexts_long = pd.read_csv(os.path.join(DATA_PATH, "processed/contexts_2024.csv"))

intv = interventions.reset_index()   # interventions frame (decision_id, participant_id, hour_utc, ...)
streams = [c for c in contexts_long.columns if c not in ('participant_id', 'context_time_utc', 'context_time_local', 'device_type')]

feats = contexts_between_decisions(contexts_long.reset_index(), intv, streams=streams)


In [ ]:
# export
for device_type, df in feats.groupby('device_type'):
    df.to_csv(os.path.join(DATA_PATH, f"processed/contexts_{device_type}_2024.csv"))
del df, device_type, feats
gc.collect()

## Combine

In [ ]:
# TODO: iterate through device types
df1 = pd.read_csv(os.path.join(DATA_PATH, "processed/interventions_2024.csv"))\
    .set_index(['participant_id', 'decision_date'])
df2 = pd.read_csv(os.path.join(DATA_PATH, "processed/contexts_2024.csv"))\
    .set_index(['participant_id', 'decision_date'])
# TODO: determine whether join should be left or inner
df = df1.merge(df2, how='left', left_index=True, right_index=True)